<a id="top"></a>

# Lab L0210: the single-step task catalog

```yaml
title: "Lab L0210: the single-step task catalog"
keywords: task shapes, classification, ranking, constrained generation, summarization, validators, lab
estimated duration: 35
```

> **Lesson:** L02. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L02/objectives.md). Solutions: `L0210_lab_solutions.ipynb`. Problems 1, 2, and 4 are pure Python; Problem 3 ends with one **live** call -- set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running it, and clear that output before committing.
>
> **After this lab you can:** name a task by its shape, write the prompt that asks for its output contract, and write the **validator** that checks the model actually honored it.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- Classification: prompt + validator](#2-problem-1--classification-prompt--validator)
- [3. Problem 2 -- Ranking: validate the set is preserved](#3-problem-2--ranking-validate-the-set-is-preserved)
- [4. Problem 3 -- Constrained generation: count + length, then a live call](#4-problem-3--constrained-generation-count--length-then-a-live-call)
- [5. Problem 4 -- Summarization: a guardrail check](#5-problem-4--summarization-a-guardrail-check)
- [6. Problem 5 -- Name the shape](#6-problem-5--name-the-shape)
- [7. Self-check](#7-self-check)

## 1. Setup

In [ ]:
import json
import re
from typing import Any

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient


# A defensive JSON parser, given (you built one like it in the L0206 lab). Problems 1-3 reuse it.
def parse_json(reply: str) -> Any:
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        pass
    match = re.search(r"[\[{].*[\]}]", reply, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    raise ValueError(f"Could not parse JSON from reply:\n{reply!r}")


print("setup ready")

[↑ Back to top](#top)

## 2. Problem 1 -- Classification: prompt + validator

Classification sorts an input into a **fixed label set**. Write two functions:

- `classify_prompt(labels)` -- return an instruction telling the model to pick **exactly one** of `labels` and return **only** the label.
- `validate_label(label, allowed)` -- return a list of problems (empty == valid). The check: an **enum is a contract, not a constraint**, so confirm `label` is in `allowed`.

In [ ]:
def classify_prompt(labels: list[str]) -> str:
    # TODO: return an instruction that says "pick exactly one of {labels}, return only the label".
    # Hint: ", ".join(labels).
    raise NotImplementedError("your code here")


def validate_label(label: str, allowed: list[str]) -> list[str]:
    # TODO: return [] if label is in allowed, else a one-item list describing the problem.
    raise NotImplementedError("your code here")


ALLOWED = ["P0-bug", "feature-discussion", "docs-clarity", "billing-question"]
print(classify_prompt(ALLOWED))
print(validate_label("P0-bug", ALLOWED))  # expect []
print(validate_label("bug", ALLOWED))  # expect a not-in-set problem

[↑ Back to top](#top)

## 3. Problem 2 -- Ranking: validate the set is preserved

Ranking returns the candidate **ids** in a new order. The model can silently drop, duplicate, or invent an id -- so the validator is the point. Write `validate_ranking(ranked_ids, candidate_ids)`: return a list of problems (empty == valid). A valid ranking contains **every candidate id exactly once**.

In [ ]:
def validate_ranking(ranked_ids: list[int], candidate_ids: list[int]) -> list[str]:
    # TODO: flag duplicates (len vs len(set)), and flag a changed id set
    # (set(ranked_ids) != set(candidate_ids)). Return a list of problems; [] means valid.
    raise NotImplementedError("your code here")


CANDS = [1, 2, 3, 4]
print(validate_ranking([3, 1, 4, 2], CANDS))  # expect []  (a valid permutation)
print(validate_ranking([3, 1, 2], CANDS))  # expect a problem (4 dropped)
print(validate_ranking([3, 1, 2, 2], CANDS))  # expect a problem (2 duplicated, 4 missing)

[↑ Back to top](#top)

## 4. Problem 3 -- Constrained generation: count + length, then a live call

Constrained generation must honor a **checkable rule**. Write `validate_items(items, n, max_words)`: return problems if there are not exactly `n` items, or any item exceeds `max_words` words. Then the given code runs one **live** call and validates the result -- a constraint you don't check is one the model is free to miss.

In [ ]:
def validate_items(items: list[str], n: int, max_words: int) -> list[str]:
    # TODO: flag a wrong count (len(items) != n) and any item with more than max_words words.
    # Hint: line.split(). Return a list of problems; [] means valid.
    raise NotImplementedError("your code here")


# Offline checks first:
print(validate_items(["a b", "c", "d e f", "g", "h"], 5, 8))  # expect []
print(validate_items(["a", "b", "c"], 5, 8))  # expect a wrong-count problem
print(validate_items(["one two three four five six seven eight nine"], 1, 8))  # over-length

# Now one LIVE call: generate exactly N subject lines, each <= MAX_WORDS words, then validate.
N, MAX_WORDS = 5, 8
client: PotatoLLMClient = AnthropicClient()
reply = client.chat(
    [
        Message.system(
            f"Generate exactly {N} onboarding-email subject lines, each at most {MAX_WORDS} "
            "words. Return only a JSON array of strings."
        ),
        Message.user("Product: a habit-tracking app for students."),
    ],
    max_tokens=200,
    temperature=0.7,
).text
generated = parse_json(reply)
print("generated:", generated)
print("problems :", validate_items(generated, N, MAX_WORDS) or "none")

[↑ Back to top](#top)

## 5. Problem 4 -- Summarization: a guardrail check

Summarization has no single right answer, so you validate with **guardrails, not equality**. Write `validate_summary(summary, source, max_words, must_include)`: return problems if the summary (a) has more than `max_words` words, (b) is **not shorter** (in words) than the `source`, or (c) drops any required term in `must_include` (case-insensitive -- a cheap proxy for "kept the load-bearing facts"). Return a list; empty == valid.

In [ ]:
def validate_summary(
    summary: str, source: str, max_words: int, must_include: list[str]
) -> list[str]:
    # TODO: return problems if (a) len(summary.split()) > max_words,
    # (b) the summary is not shorter than the source in words, or
    # (c) any term in must_include is absent (case-insensitive). [] means valid.
    raise NotImplementedError("your code here")


SOURCE = (
    "The Q3 release shipped offline mode, cut cold-start time by 40%, and fixed the sync "
    "bug that occasionally duplicated tasks. Offline-mode adoption hit 30% in two weeks."
)
# Offline crafted checks: a good summary, then two that trip a guardrail.
print(
    validate_summary(
        "Q3 shipped offline mode and fixed the sync bug.", SOURCE, 20, ["offline", "sync"]
    )
)  # expect []
print(validate_summary("A tiny note.", SOURCE, 20, ["offline", "sync"]))  # missing required terms
print(
    validate_summary(SOURCE + " and more text besides.", SOURCE, 20, ["offline"])
)  # too long / not shorter

[↑ Back to top](#top)

## 6. Problem 5 -- Name the shape

For each task, name which single-step **shape** it is (extraction, classification, ranking, constrained generation, or summarization) and the **one thing you'd validate**. Fill in the table.

| Task | Shape | One thing to validate |
| --- | --- | --- |
| Pull the invoice number and total from a PDF's text | _TODO_ | _TODO_ |
| Order 10 support tickets from most to least urgent | _TODO_ | _TODO_ |
| Write 3 tweet variants, each under 280 characters | _TODO_ | _TODO_ |
| Turn a research abstract into a plain-language paragraph | _TODO_ | _TODO_ |

## 7. Self-check

Compare against `L0210_lab_solutions.ipynb`. Done when `validate_label` flags an out-of-set label, `validate_ranking` catches a dropped and a duplicated id, `validate_items` catches a wrong count and an over-length item, `validate_summary` catches an over-long summary and a missing required term, and you can name the shape of each task in Problem 5.

[↑ Back to top](#top)